# E4 — Coupled $\phi^4$ / Ginzburg–Landau chain (24D): coherent flips via moment-exact homogeneous jumps

$q_i\in\mathbb R^2$, $i=0,\dots,11$, periodic, $N_s=12$, $\delta = 1/N_s$, $d=24$, $\kappa=2.5$:
$$V(q) = \frac{\kappa}{2\delta}\sum_i \|q_{i+1}-q_i\|^2 + \delta\sum_i W(q_i),\qquad
\boxed{W(x,y) = (x^2-1)^2 + (y^2-1)^2 - 0.05\,xy + 0.03\,x + 0.06\,y}$$
The small tilt terms split the four phases without destroying any well. *(The legacy parameters from `experiments/` are **not** used: their $\eta = 0.467$ term reduced the $-+$ well's escape barrier to $0.0026$ and its soft Hessian eigenvalue to $0.335$ — the well was all but destroyed. $\eta$ existed to break parallel jump-edge directions for a graph-family ablation not run here.)*

For a homogeneous field $q_i\equiv v$ the gradient energy vanishes and $V = W(v)$: **the coherent barrier equals the barrier of $W$**. Coherent flip vs nucleation: the kink energy is $\sigma = \int_{-1}^1\sqrt{2\kappa(1-x^2)^2}dx = \tfrac43\sqrt{2\kappa} = 2.98$, so a periodic wall pair costs $5.96 \gg 1.0$ — **the coherent flip is the minimum-energy path**, which is precisely what makes the homogeneous-shift jump law (and its moment-exact score) the right choice.

In [ ]:
EXPERIMENT = "coupled_phi4"
import os, sys, math, time, json
sys.path.insert(0, os.path.abspath(".."))
from src.gpu_guard import select_gpu
select_gpu(int(os.environ.get("JCP_GPU", "4")))
import torch
assert torch.cuda.device_count() == 1, "GPU guard must mask to exactly one device"
torch.set_default_dtype(torch.float64)
import numpy as np
import pandas as pd

from src import config as C
from src.experiments import build_e4, make_sampler_factory, make_metrics
from src.runner import (run_experiment, run_one, refine_dt, quadrature_refinement,
                        write_timeseries_csv, write_summary_csv, write_manifest,
                        ula_first_passage, hardware_manifest)
from src.samplers import tune_ladder
from src.certificate import make_phi_family, certificate_grid, certificate_importance
from src.plotting import make_all_figures, apply_style

DEV = "cuda"
RESULTS = os.path.abspath(os.path.join("..", "results", EXPERIMENT))
FIGURES = os.path.abspath(os.path.join("..", "figures", EXPERIMENT))
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
exp = build_e4(device=DEV, basin_cache=os.path.join(RESULTS, "basin_map.npz"))
cfg = exp.cfg
print(f"experiment={cfg.name}  d={cfg.d}  N={cfg.n_particles}  T={cfg.T}  dt0={cfg.dt}")
print(f"beta={cfg.beta}  eps={cfg.eps}  lambda={cfg.lam}  seeds={cfg.seeds}")
print(hardware_manifest())

## The model

Verified coherent minima of $W$ (asserted to 4 decimals below):

| phase | minimum | $W$ | escape barrier | Laplace mass |
|---|---|---|---|---|
| $--$ | $(-1.0099,-1.0135)$ | $-0.1412$ | 1.082 | 0.583 |
| $-+$ | $(-0.9976,\ 0.9860)$ | $+0.0792$ | 0.892 | 0.106 |
| $+-$ | $(\ 0.9898,-1.0013)$ | $+0.0196$ | 0.921 | 0.169 |
| $++$ | $(\ 1.0025,\ 0.9988)$ | $+0.0400$ | 0.990 | 0.142 |

$\beta\times\min$ barrier $= 7.14$. Stiffness: $\max\lambda(\nabla^2 V)\approx 4\kappa/\delta = 120$, so $\Delta t < 0.017$. Protocol: $N=1000$, $T=100$, $\Delta t_0=0.002$, box $[-2,2]^{24}$, init at the $--$ coherent state $+\,0.05\,\xi$. Partition ($K=4$): basin map of $W$ at the mean order parameter $\bar q = \tfrac1{N_s}\sum_i q_i$.

**References, not ground truth:** (i) the harmonic (Laplace) mixture — phase $k$ with weight $\propto e^{-\beta W(v_k)}/\sqrt{\det H_k}$ ($H_k$ the full $24\times24$ Hessian at the coherent minimum), fluctuations $\mathcal N(0,\varepsilon H_k^{-1})$; (ii) a long PT chain as a cross-check. Both are labelled as references below.

In [ ]:
from src.potentials import (PHI4_MINIMA, PHI4_ESCAPE_BARRIERS, PHI4_LAPLACE_MASSES,
                            phi4_W, phi4_W_grad, newton_refine)
V2 = exp.extras["minima_2d"]
phases = exp.extras["phases"]
for i, ph in enumerate(phases):
    v_tab, W_tab = PHI4_MINIMA[ph]
    v = V2[i]
    W = phi4_W(v.unsqueeze(0))[0].item()
    assert abs(v[0].item() - v_tab[0]) < 5e-5 and abs(v[1].item() - v_tab[1]) < 5e-5
    assert abs(W - W_tab) < 5e-4
    print(f"{ph}: v = ({v[0].item():+.4f}, {v[1].item():+.4f})  W = {W:+.4f}  [table {v_tab}, {W_tab}]")

saddle_guesses = [(-1.0, 0.0), (1.0, 0.0), (0.0, -1.0), (0.0, 1.0)]
saddles = [(newton_refine(phi4_W_grad, torch.tensor(sg, device=DEV))) for sg in saddle_guesses]
sW = [phi4_W(s.unsqueeze(0))[0].item() for s in saddles]
adj = {"--": [0, 2], "-+": [0, 3], "+-": [1, 2], "++": [1, 3]}
for i, ph in enumerate(phases):
    Wm = phi4_W(V2[i].unsqueeze(0))[0].item()
    bar = min(sW[j] - Wm for j in adj[ph])
    assert abs(bar - PHI4_ESCAPE_BARRIERS[ph]) < 2e-3, (ph, bar)
    print(f"{ph}: escape barrier {bar:.4f} [table {PHI4_ESCAPE_BARRIERS[ph]}]")
print(f"beta * min barrier = {C.BETA * min(PHI4_ESCAPE_BARRIERS.values()):.2f}")

sigma_kink = exp.pot.kink_energy()
print(f"kink energy sigma = (4/3)sqrt(2 kappa) = {sigma_kink:.3f}; "
      f"periodic wall pair costs {2*sigma_kink:.2f} >> 1.0 coherent barrier"
      " -> the coherent flip is the minimum-energy path")

print("Laplace masses:", np.round(exp.p_star.cpu().numpy(), 3),
      " [table:", list(PHI4_LAPLACE_MASSES.values()), "]")
for i, ph in enumerate(phases):
    assert abs(exp.p_star[i].item() - PHI4_LAPLACE_MASSES[ph]) < 5e-3, ph

g = torch.Generator(device=DEV); g.manual_seed(0)
barrier_report = ula_first_passage(exp.pot, exp.box, exp.init_fn(cfg.n_particles, g),
                                   exp.exit_committed, cfg.dt, int(cfg.T/cfg.dt), C.EPS, g)
barrier_report["kramers_tau_langer"] = exp.kramers_tau
print("ULA first-passage out of the -- basin:", barrier_report)
print(f"(24D Langer estimate over the coherent saddle: tau ~ {exp.kramers_tau:.0f})")

## Jump law and the moment-exact score

Homogeneous phase-to-phase shifts on the complete graph over the 4 minima (6 undirected $\to$ 12 directed atoms): $r_a = \mathbf 1_{N_s}\otimes(v_j-v_i)$, $w_a = 1/12$, $h = 0.1\min_a\|r_a\|$, $\lambda=1$ (a shell-thickened homogeneous shift is still homogeneous).

The periodic gradient energy is **exactly invariant** under $q_i \mapsto q_i - d$, so
$$V(q-r)-V(q) = \delta\sum_i\big[W(q_i-d)-W(q_i)\big]$$
is a fixed polynomial in $(d_x,d_y)$ — e.g. $((x-d_x)^2-1)^2-(x^2-1)^2 = -4d_xx^3+6d_x^2x^2+(4d_x-4d_x^3)x+d_x^4-2d_x^2$ — whose coefficients are the per-particle moments $\sum_i x_i, \sum_i x_i^2, \sum_i x_i^3$ (and $y$ analogues). The moments cost $O(N_s)$ once per step; all $12\times Q_\rho\times Q_\theta$ energy deltas are then $O(1)$ arithmetic each. **No lattice sweeps.** Validated against the direct lattice energy difference to $10^{-13}$ absolute in `tests/test_score.py`.

In [ ]:
print("12 directed homogeneous atoms; per-site shifts (dv):")
print(np.round(exp.law.atoms[:, :2].cpu().numpy(), 4))
print("h =", round(exp.extras["h"], 4), " ||r_a|| =",
      np.round(exp.law.atoms.norm(dim=1).cpu().numpy(), 3))

from src.jumps import gauss_legendre_01
DEFAULT_QUAD = dict(q_theta=C.Q_THETA, q_rho=C.Q_RHO)
phis = make_phi_family(24, exp.extras["means24"][0].tolist(), 1.5, DEV, n_phi=4)

def cert_e4(q_theta, q_rho):
    theta, w_theta = gauss_legendre_01(q_theta, DEV)
    shifts, logw = exp.law.quadrature_shifts(q_rho)
    shifts_j, logw_j = exp.law.quadrature_shifts(64)   # fine continuous-nu J side
    return certificate_importance(exp.pot, shifts, logw, theta, w_theta,
                                  cfg.lam, cfg.beta, phis, exp.extras["laplace"],
                                  n_samples=200_000,
                                  nu_shifts_jump=shifts_j, nu_logw_jump=logw_j)

## Target preservation: the stationarity identity $(\star)$

The LSC-CP generator is
$$\mathcal A f = \big[-\nabla V + S_{\nu,\beta}\big]\cdot\nabla f + \varepsilon\,\Delta f + \lambda\!\int\!\big[f(x+r)-f(x)\big]\nu(dr),$$
with $\nu$ a **probability** measure and
$$S_{\nu,\beta}(x) = -\lambda \int \nu(dr)\; r \int_0^1 \exp\!\Big[-\beta\big(V(x-\theta r) - V(x)\big)\Big]\, d\theta .$$
Raw CP is the same generator with $S \equiv 0$.

Write $p = e^{-\beta V}/Z$. The overdamped part is $\pi$-reversible, so invariance of $\pi$ is equivalent to
$$\int S\cdot\nabla\varphi \, d\pi + \int J\varphi \, d\pi = 0 \qquad \forall\, \varphi \in C_c^\infty . \tag{$\star$}$$

**Jump term.** Shift the integration variable and apply the fundamental theorem of calculus along $\theta \mapsto y - \theta r$:
$$\int J\varphi\,d\pi = \lambda\!\int\!\nu(dr)\!\int\!\varphi(y)\big[p(y-r)-p(y)\big]dy = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(y)\,r\!\cdot\!\nabla p(y-\theta r)\,dy\,d\theta.$$

**Drift term.** $S(x) = -\lambda\int\nu(dr)\,r\int_0^1 \frac{p(x-\theta r)}{p(x)}d\theta$ (identical to the boxed formula since $p \propto e^{-\beta V}$), so integrating by parts in $x$:
$$\int S\cdot\nabla\varphi\,d\pi = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\big(r\!\cdot\!\nabla\varphi(x)\big) p(x-\theta r)\,dx\,d\theta = +\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(x)\,r\!\cdot\!\nabla p(x-\theta r)\,dx\,d\theta.$$

The two cancel identically. **Target preservation is unconditional in $\nu$** — any finite-activity jump law works; only the *speed* depends on $\nu$.

### The measured certificate $\mathcal R(\varphi)$

For smooth bounded test functions (products of tanh ridges) we report
$$\mathcal R(\varphi) = \frac{\big|\int S_{\nu,\beta}\!\cdot\!\nabla\varphi\,d\pi + \int J_\nu\varphi\,d\pi\big|}{\big|\int J_\nu\varphi\,d\pi\big|},$$
zero in exact arithmetic; the measured value is the combined defect of the $\theta$/$\rho$ quadratures. Two implementation notes, both load-bearing:

1. **The integration domain extends a full jump length beyond the target's effective support.** Order-one contributions to $(\star)$ live where $\pi$ is tiny and $S$ is enormous; a deliberately tight box produces a large residual (demonstrated below and regression-tested).
2. **The drift integrand $p\,S\cdot\nabla\varphi$ is assembled in log space** from the score's $(M, v)$ parts as $\exp(-\beta V + M)\,v\cdot\nabla\varphi$: in linear fp64 arithmetic $p$ underflows exactly where $\|S\|$ is astronomical, silently dropping those order-one far-field contributions. The residual uses the *uncapped* $M$; the deployed drift caps $M$ at $M_{\max}=600$, but because taming saturates (the tamed step tends to $-v/\|v\|$), the deployed tamed step differs from the uncapped one by $O(e^{-M_{\max}})$ — that saturation defect is reported alongside $\mathcal R$ and is $\lesssim 10^{-250}$ here.

A useful exact identity (change of variables $x \to x+\theta_p r$ in the drift term): for the *implemented* quadrature score,
$$\int S\cdot\nabla\varphi\,d\pi + \int J\varphi\,d\pi \;=\; \lambda\,\mathbb E_\pi\!\int\!\nu(dr)\Big[\varphi(x+r)-\varphi(x) - \sum_p \hat w_p\, r\cdot\nabla\varphi(x+\theta_p r)\Big],$$
i.e. the residual is **independent of $V$** and equals the $\theta$-quadrature error on the smooth test-function integrand. This is why moderate pointwise errors of the Gauss–Legendre rule on the stiff factor $e^{\beta\Delta V}$ do not translate into a weak (distributional) defect of the sampled law.

In 24D a grid is infeasible; the residual uses the shifted-form identity above with self-normalised importance sampling from the Laplace mixture. This is exactly equivalent to the deployed quadrature score provided the $M_{\max}$ cap never fires on the sampled region — asserted below (the max log-magnitude there is $\approx 15 \ll 600$).

In [ ]:
cert_report = cert_e4(**DEFAULT_QUAD)
for i in range(len(phis)):
    print(f"  phi_{i}: R = {cert_report[f'phi_{i}']['residual']:.3e}")
print(f"max R = {cert_report['max_residual']:.3e}")
assert cert_report["max_residual"] < 1e-6

score = exp.make_score(**DEFAULT_QUAD)
g = torch.Generator(device=DEV); g.manual_seed(11)
Mv, _ = score.log_parts(exp.extras["laplace"].sample(100_000, g))
print(f"max log score magnitude on the sampled region: {Mv.max().item():.1f} << 600 (cap never fires)")
cert_report["max_log_magnitude_on_support"] = float(Mv.max().item())

## The seven methods

All methods share one **taming policy**: the same map $b \mapsto b/(1+\Delta t\,\|b\|)$ is applied to every method's drift (ULA, the MALA proposal, FLA, the BAOAB force, raw CP, LSC-CP). Tamed MALA is still exact because the proposal density $q(y|x) = \mathcal N(y;\, x + \Delta t\, b_{\rm tamed}(x),\, 2\varepsilon\Delta t\, I)$ is used consistently in both directions of the MH ratio; asymmetric taming would make taming a hidden variable in the comparison.

**1. ULA.** $X \leftarrow X + \Delta t\,\mathrm{tame}(-\nabla V) + \sqrt{2\varepsilon\Delta t}\,\xi$.

**2. MALA.** With $\nabla\log\pi = -\beta\nabla V$, the proposal $Y = X - \tfrac{h\beta}2\nabla V(X) + \sqrt h\,\xi$ matches the ULA step iff $\tfrac{h\beta}{2} = \Delta t$ **and** $h = 2\varepsilon\Delta t$. Both conditions coincide:
$$\boxed{h = \frac{2\Delta t}{\beta} = 2\varepsilon\Delta t = \Delta t/4 \quad\text{at }\beta=8.}$$
Log-acceptance
$$\log\alpha = -\beta[V(Y)-V(X)] - \frac{1}{2h}\Big[\|X - \mu(Y)\|^2 - \|Y-\mu(X)\|^2\Big],\qquad \mu(z) = z + \Delta t\,\mathrm{tame}(-\nabla V(z)),$$
accepted elementwise. Proposals are **never clipped before the accept step** (that silently breaks exactness); out-of-box proposals are auto-rejected, which is valid MH for the box-restricted target. Expect acceptance $\approx 1$ — the honest message: **rejection does not cure metastability**.

**3. FLA / FLMC** (Şimşekli, ICML 2017, §3.3). $X \leftarrow X + \Delta t\,\mathrm{tame}(-c_\alpha \nabla U) + \Delta t^{1/\alpha}\,\xi^{(\alpha)}$ with $U = \beta V$, $c_\alpha = \Gamma(\alpha-1)/\Gamma(\alpha/2)^2$, $\alpha = 1.7$; per-coordinate $S\alpha S(1)$ noise by Chambers–Mallows–Stuck. **No tail clipping** — a truncated stable is not stable. FLA is the *uncorrected nonlocal* comparator: heavy tails cross barriers, but the invariant law is not $\pi$.

**4. Kinetic Langevin (BAOAB).** It is **not HMC** (no accept/reject; carries $O(\Delta t^2)$ configurational bias). Unit mass, $\gamma = 1$; the O-step coefficient is the exact OU solution: $dp = -\gamma p\,dt + \sqrt{2\gamma\varepsilon}\,dW$ gives $\mathrm{Var} = 2\gamma\varepsilon\int_0^{\Delta t}e^{-2\gamma s}ds = \varepsilon(1-e^{-2\gamma\Delta t})$ by Itô isometry. The trailing force is cached as the next step's leading B (one gradient per step).

**5. Parallel tempering.** MALA-within-replica at $\beta_k = \beta\, r^{k-1}$, replica $k$ using $h_k = 2\Delta t/\beta_k$ (same tamed drift step for every replica; only the noise scale differs). Adjacent swaps every $n_{\rm swap}$ steps (alternating parity); the joint target is $\prod_k \pi_k$ and the swap is a deterministic involution, so
$$\alpha_{\rm swap} = \min\Big\{1,\ \exp\big[(\beta_i - \beta_{i+1})\big(V(x_i) - V(x_{i+1})\big)\big]\Big\}.$$
$V$ values are cached by MALA, so swaps are free in evaluation count. $K$ is tuned so the mean swap acceptance lands in $[0.2, 0.4]$. The replica index is a batch dimension, state $(K, N, d)$; metrics use the cold replica only; **wall-clock includes all $K$ replicas**.

**6/7. Raw CP and LSC-CP.** Identical time discretisation,
$$X^{(1)} = X_n + \Delta t\,\frac{b(X_n)}{1+\Delta t\,\|b(X_n)\|} + \sqrt{2\varepsilon\Delta t}\;\xi_n,\qquad X_{n+1} = X^{(1)} + \sum_{k=1}^{N_n} A_k,$$
$N_n \sim \mathrm{Poisson}(\lambda\Delta t)$, $A_k \stackrel{iid}\sim \nu$; $b = -\nabla V$ (raw CP) or $b = -\nabla V + S_{\nu,\beta}$ (LSC-CP). The jump stream is a dedicated generator seeded identically for both methods, so their jump times and increments are **pathwise identical** (verified in `tests/test_samplers.py`), not merely equal in law.

In [ ]:
# PT ladder: geometric in beta, K tuned so mean swap acceptance is in [0.2, 0.4]
gen = torch.Generator(device=DEV); gen.manual_seed(0)
x0_pilot = exp.init_fn(min(512, cfg.n_particles), gen)
pt_betas, ladder_info = tune_ladder(exp.pot, x0_pilot, cfg.dt, exp.box,
                                    C.BETA, exp.pt_beta_min, pilot_steps=600)
print(f"PT ladder: K={ladder_info['K']}  r={ladder_info['r']:.4f}  "
      f"beta_K={pt_betas[-1].item():.4f}  swap acceptance={ladder_info['swap_acceptance']:.3f}")
print("tuning history {K: acceptance}:", ladder_info["history"])

## Reference, partition, metrics, bias floors

Reference sample size equals the run's $N$; metrics are evaluated at every checkpoint (cadence fixed in $t$, identical across methods).

* **$W_2$**: exact in 1D (sorted coupling); **sliced** $W_2$ for $d\ge2$ with $L=200$ projections drawn once from a fixed seed and reused across all times and methods (its bias floor decays like $N^{-1/2}$, not $N^{-1/d}$).
* **TV** (occupancy, on the partition): $\tfrac12\sum_k|\hat p_k - p^\star_k|$ — a **lower bound** on the full TV.
* **MMD**: Gaussian kernel, bandwidth **frozen once** by the median heuristic on the reference sample (per-frame bandwidths would make curves non-comparable); biased V-statistic $\widehat{\mathrm{MMD}}_b^2 = \|\mu_X-\mu_Y\|_{\mathcal H}^2 \ge 0$.
* **EMC** $= e^{H(\hat p)}/K$: plotted with a horizontal line at the target $e^{H(p^\star)}/K$; EMC $=1$ is optimal only for uniform $p^\star$, and deviation in *either* direction is error.
* **EJS**: base-2 Jensen–Shannon divergence between $\hat p$ and $p^\star$ (Blessing et al., arXiv:2406.07423, App. A.3), bounded in $[0,1]$, quadratic near the target, so it stays informative where TV saturates.
* **Bias floors** (mandatory): each metric between two independent reference samples of size $N$, 20 replicates; dashed line on every panel. Without this, every plateau is uninterpretable.
* **Nonfinite fraction**: logged per method per checkpoint; must be identically zero — metrics on survivors only would be survivorship bias, so nothing is ever filtered.

**Coverage vs correctness, once:** EMC measures *coverage*, TV/EJS measure *correctness*. Raw CP, whose invariant law is not $\pi$, over-flattens — driving EMC toward 1 (above its target line) while TV and EJS stay bad. That pairing *is* the raw-CP-vs-LSC-CP story.

$W_2$ and MMD are computed on the 2D mean order parameter $\bar q$; occupancy metrics on the $K=4$ basin map of $W$ at $\bar q$. The Laplace mixture supplies both the reference sample and $p^\star$ — it is a **reference, not ground truth** — so a long PT chain is run below as an independent cross-check of the phase masses.

In [ ]:
metrics_fn, floors, aux = make_metrics(exp, cfg.n_particles)
emc_target = exp.emc_target
print("p_star:", np.round(exp.p_star.cpu().numpy(), 6))
print("EMC target line: %.4f" % emc_target)
print("MMD bandwidth (median heuristic on reference, frozen):", round(aux["bandwidth"], 4))
print("bias floors (mean +- std over 20 replicate pairs):")
for k, v in floors.items():
    print(f"  {k:>12s}: {v['mean']:.5f} +- {v['std']:.5f}")

# PT cross-check of the Laplace phase masses (reference vs reference)
t0 = time.time()
gen_x = torch.Generator(device=DEV); gen_x.manual_seed(4242)
from src.samplers import ParallelTempering
pt_x = ParallelTempering(exp.pot, exp.init_fn(1000, gen_x), cfg.dt, pt_betas, gen_x, exp.box)
n_x = int(round(300.0 / cfg.dt))
for _ in range(n_x):
    pt_x.step()
from src.metrics import occupancy
p_pt = occupancy(exp.labels_fn(pt_x.positions()), 4).cpu().numpy()
print(f"long PT chain (T=300, cold replica) phase masses:   {np.round(p_pt, 3)}")
print(f"harmonic (Laplace) reference phase masses:          "
      f"{np.round(exp.p_star.cpu().numpy(), 3)}  ({time.time()-t0:.0f}s)")
pt_crosscheck = p_pt.tolist()

## $\Delta t$ refinement and production

Declared $\Delta t$ selection rule, applied uniformly to every experiment (reported in the SI): **the largest $\Delta t$ on a dyadic grid at which every method's terminal value of every metric is within 5% of its $\Delta t/2$ value.** Three statistical guards make the rule meaningful at a single refinement seed: differences are measured relative to $\max(|m_{\Delta t/2}|,\ \text{bias floor})$; when *both* values sit inside the floor band (floor mean $+\,3$ s.d.) they are declared in agreement; and differences within $4\times$ the floor s.d. — the natural unit of single-run metric sampling noise at this $N$ — are likewise noise, not discretisation bias. The same guards apply to the quadrature-refinement comparison.

**One declared exception:** the two comparators whose invariant law is not $\pi$ — FLA and raw CP — do not gate the $\Delta t$ selection. Their terminal values measure an intrinsic bias, not convergence to a target, so there is no $\Delta t$ at which they *should* stabilise to 5% at single-seed resolution (empirically FLA's density error drifts monotonically under refinement, and raw CP's bias wobbles at its own sampling noise); demanding stability from them would refine $\Delta t$ forever. Both still run at the shared chosen $\Delta t$, and their deviations across the dyadic grid are recorded in the refinement table for transparency. The gate is carried by the five $\pi$-targeting methods.

Production protocol: 5 seeds $\times$ 7 methods, run **sequentially** (never batched) so per-run wall-clock is meaningful; all methods share $x_0$ per seed; 20 untimed warm-up steps absorb allocator/JIT effects; `torch.cuda.synchronize()` brackets every timed region, and the timer covers sampler work only.

In [ ]:
def run_terminal_lsc(**quad):
    f = make_sampler_factory(exp, cfg.dt, pt_betas, score_kwargs=quad)
    n_ = int(round(cfg.T / cfg.dt))
    r_, _ = run_one("LSC-CP", 0, f, n_, n_, cfg.dt, metrics_fn, exp.pot, quiet=True)
    return {k: r_[-1][k] for k in ("W2", "TV", "MMD", "EMC", "EJS")}

settings = [dict(q_theta=qt, q_rho=qr) for qt in (8, 16, 32) for qr in (4, 8, 16)]
CHOSEN_QUAD, quad_table = quadrature_refinement(
    settings, run_terminal_lsc, lambda **s: cert_e4(**s)["max_residual"], floors)
print("chosen production quadrature:", CHOSEN_QUAD)
display(pd.DataFrame(quad_table).round(6))
if CHOSEN_QUAD != DEFAULT_QUAD:
    cert_report = cert_e4(**CHOSEN_QUAD)
    print("certificate re-evaluated at chosen orders: max R =",
          f"{cert_report['max_residual']:.3e}")
    assert cert_report["max_residual"] < 1e-6

In [ ]:
MAIN_METRICS = ["W2", "TV", "MMD", "EMC", "EJS"]

def run_terminal_all(dt_):
    n_ = int(round(cfg.T / dt_))
    factory = make_sampler_factory(exp, dt_, pt_betas, score_kwargs=CHOSEN_QUAD)
    out = {}
    for m in C.METHODS:
        rows_, _ = run_one(m, 0, factory, n_, n_, dt_, metrics_fn, exp.pot, quiet=True)
        out[m] = {k: rows_[-1][k] for k in MAIN_METRICS}
    print(f"  refine_dt: finished pass at dt={dt_}", flush=True)
    return out

dt_final, dt_table = refine_dt(run_terminal_all, cfg.dt, floors, exclude=("FLA", "CP"))
print("chosen dt:", dt_final)
for row in dt_table:
    print(row)

n_steps = int(round(cfg.T / dt_final))
steps_per_ck = max(1, n_steps // C.N_CHECKPOINTS)
factory = make_sampler_factory(exp, dt_final, pt_betas, score_kwargs=CHOSEN_QUAD)
t0 = time.time()
rows, method_info = run_experiment(C.METHODS, cfg.seeds, factory, n_steps,
                                   steps_per_ck, dt_final, metrics_fn, exp.pot)
print(f"production total: {time.time()-t0:.0f}s")
worst_nonfinite = max(r["nonfinite_frac"] for r in rows)
assert worst_nonfinite == 0.0, worst_nonfinite
print("nonfinite fraction: identically zero across all methods/checkpoints")

## Figures

In [ ]:
fig_metrics = ("W2", "TV", "MMD", "EMC", "EJS")
written = make_all_figures(rows, FIGURES, floors, emc_target, metrics=fig_metrics)
print(f"{len(written)} figures x 3 formats (.pdf/.png 600dpi/.eps) + captions -> {FIGURES}")

# grid display for inspection (saved files above are one-figure-per-file)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
n_m = len(fig_metrics)
fig, axes = plt.subplots(n_m, 2, figsize=(11, 3.2 * n_m))
for i, metric in enumerate(fig_metrics):
    for j, tag in enumerate(("vs_time", "vs_wallclock")):
        ax = axes[i, j] if n_m > 1 else axes[j]
        ax.imshow(mpimg.imread(os.path.join(FIGURES, f"{metric}_{tag}.png")))
        ax.set_axis_off()
plt.tight_layout(); plt.show()

## CSV emission and summary

In [ ]:
ts_path = os.path.join(RESULTS, "metrics_timeseries.csv")
write_timeseries_csv(rows, ts_path)
summary_metrics = MAIN_METRICS + ["nonfinite_frac"]
summary = write_summary_csv(rows, C.METHODS, cfg.seeds, summary_metrics,
                            method_info, floors, os.path.join(RESULTS, "summary.csv"))

manifest = dict(
    experiment=EXPERIMENT,
    config=dict(d=cfg.d, N=cfg.n_particles, T=cfg.T, dt0=cfg.dt, dt=dt_final,
                beta=cfg.beta, eps=cfg.eps, lam=cfg.lam, seeds=list(cfg.seeds),
                n_checkpoints=C.N_CHECKPOINTS, warmup_steps=C.N_WARMUP_STEPS),
    quadrature=dict(chosen=CHOSEN_QUAD, table=quad_table),
    dt_refinement=[{k: (str(v) if isinstance(v, tuple) else v) for k, v in row.items()}
                   for row in dt_table],
    pt_ladder={k: v for k, v in ladder_info.items()},
    certificate=cert_report,
    bias_floors=floors,
    barrier_verification=barrier_report,
    method_info={m: {k: v for k, v in mi.items() if isinstance(v, (int, float))}
                 for m, mi in method_info.items()},
    hardware=hardware_manifest(),
    pt_phase_mass_crosscheck=pt_crosscheck,
)
write_manifest(os.path.join(RESULTS, "manifest.json"), **manifest)
print("wrote", ts_path)
from IPython.display import display
display(pd.read_csv(os.path.join(RESULTS, "summary.csv")).round(5))